# Translate `DeepPavlov/Mantis` to Spanish

MANtIS is a multi-domain conversational-search dataset built from StackExchange Q&A
threads across 14 categories (apple, askubuntu, dba, diy, electronics, english, gaming,
gis, physics, scifi, security, stats, travel, worldbuilding). Each row is a `dialog`
(list of `{message, role}` turns, `role` in `user`/`assistant`), plus a `title`,
`category`, and `dialog_time` (an ISO timestamp -- not translated).

**Scale**: this is much bigger than the earlier intent-classification notebooks --
~411k dialogue turns and ~80k titles across dev/test/train combined:

| split | dialogs | turns |
|---|---|---|
| dev | 12,041 | 62,319 |
| test | 12,064 | 62,091 |
| train | 56,221 | 286,601 |

**Register**: messages are real StackExchange posts -- often long (up to ~28k chars,
though median is ~270), technical, and frequently contain embedded HTML (`<a href=...>`,
`<img>`) and code (`<code>`/`<pre>` blocks, shell commands, error messages). Those must
be preserved verbatim, not translated -- only the surrounding natural-language prose
gets translated. The system prompt and few-shot examples below are built around that.

`SPLITS_TO_RUN` (in the config cell) defaults to all three splits, but you may want to
run `dev`+`test` first (~124k turns) and check quality/timing before committing to
`train` (~287k more) -- just edit that list.

- `gemma` — `google/gemma-4-31B-it` on `http://localhost:8088/v1`
- `qwen`  — `Qwen/Qwen3.6-27B-FP8` on `http://localhost:8000/v1`

**Setup.** This repo's `uv` environment already has `datasets`; it does not have
`openai`. Launch this notebook with the extra dependency pulled in on the fly, without
touching `pyproject.toml`:

```bash
uv run --with openai --with ipykernel jupyter lab
```

Everything is checkpointed to `translations/mantis/*.jsonl`, so the notebook is safe to
interrupt and re-run — already-translated items are skipped.


In [1]:
import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from openai import OpenAI
from tqdm.auto import tqdm


## Config

Only Spanish was requested for this dataset. Trim `SPLITS_TO_RUN` if you want to start smaller than all three splits.

In [2]:
MODELS = {
    "gemma": {"base_url": "http://localhost:8088/v1", "model": "google/gemma-4-31B-it"},
    # "qwen": {
    #     "base_url": "http://localhost:8000/v1",
    #     "model": "Qwen/Qwen3.6-27B-FP8",
    #     # Qwen3 is a hybrid-thinking model: without this it emits its chain-of-thought
    #     # as the actual response content instead of a final answer.
    #     "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    # },
}

LANGUAGES = {
    "es": "Spanish",
}

SPLITS_TO_RUN = ["dev", "test", "train"]  # edit e.g. to ["dev", "test"] to defer train

OUT_DIR = Path("translations/mantis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 64
TEMPERATURE = 0.0


In [3]:
clients = {name: OpenAI(base_url=cfg["base_url"], api_key="EMPTY") for name, cfg in MODELS.items()}

for name, client in clients.items():
    available = [m.id for m in client.models.list().data]
    print(f"{name} ({MODELS[name]['base_url']}): serving {available}")
    assert MODELS[name]["model"] in available, (
        f"{MODELS[name]['model']} not found on {name} server; available: {available}"
    )


gemma (http://localhost:8088/v1): serving ['google/gemma-4-31B-it']


## Load the dataset

In [4]:
raw = load_dataset("DeepPavlov/Mantis")
for split in raw:
    n_turns = sum(len(r["dialog"]) for r in raw[split])
    print(f"{split}: {len(raw[split])} dialogs, {n_turns} turns")


dev: 12041 dialogs, 62319 turns
test: 12064 dialogs, 62091 turns
train: 56221 dialogs, 286601 turns


In [5]:
raw["test"]["dialog"][0], raw["test"]["dialog"][1]

([{'message': 'Previously with El Capitan, an app such as smcFanControl (2.6) could furrow in an <a href="https://apple.stackexchange.com/q/294229/7985">obscure way into the OS.Has something changed in either Sierra or High Sierra? After sleeping and waking up, the same app (in the same version) now requires re-entering the sudo password, or otherwise it fails to run in the menu bar.',
   'role': 'user'},
  {'message': 'Not to my knowledge, while it is true that security improvements were made with the new (High Sierra) release, the issue you are describing seems to be unrelated to it. I tried to replicate your scenario but it works fine for me.  Things to try:Reinstall the application. Maybe best done with <a href="https://freemacsoft.net/appcleaner/" rel="nofollow noreferrer">AppCleaner to remove all associated files first.Be sure it is installed to the Applications folder on your Mac.This I believe sets certain privileges to the app which it may not get if inside other, more obscure

## Translation

Per-item translation (one request per message/title), same pattern as the earlier
notebooks -- no batching or deduplication here, since these are largely unique
conversational messages (unlike the categorical StatCan vocabulary). HTML tags,
URLs, and code blocks must survive untouched.


In [6]:
EXAMPLES = {
    "es": [
        (
            "It's hardware related. Logic board needs to be replaced",
            "Es un problema de hardware. Hay que reemplazar la placa logica",
        ),
        (
            "Verify your MacBook sn: https://selfsolve.apple.com/agreementWarrantyDynamic.do",
            "Verifica el numero de serie de tu MacBook: https://selfsolve.apple.com/agreementWarrantyDynamic.do",
        ),
        (
            "Run <code>sudo apt-get update</code> first, then reboot.",
            "Ejecuta <code>sudo apt-get update</code> primero, y despues reinicia.",
        ),
    ],
}

SYSTEM_PROMPT = (
    "You are a professional translator localizing StackExchange technical Q&A posts and "
    "chat messages (topics include Apple hardware, Ubuntu/Linux, databases, electronics, "
    "physics, security, travel, and more). Translate the message from English into "
    "{lang_name}. Keep the same meaning, tone, and register -- technical but "
    "conversational -- and produce something that reads naturally to a native "
    "{lang_name} speaker. "
    "CRITICAL: preserve ALL HTML tags (e.g. <a href=...>, <img>, <pre>, <code>) and "
    "their attributes EXACTLY as written -- do not translate URLs, do not alter tag "
    "syntax. Preserve the contents of <code>/<pre> blocks, shell commands, file paths, "
    "error messages, and variable/product names (e.g. 'MacBook', 'Ubuntu', 'SSD') "
    "UNTRANSLATED -- only translate the surrounding natural-language prose. "
    "Do not add, remove, or explain anything. "
    "Reply with ONLY the translated message: no quotes, no notes, no alternatives.\n\n"
    "Examples:\n{examples_block}"
)


def build_examples_block(lang_code):
    lines = [f"EN: {en}\n{lang_code.upper()}: {es}" for en, es in EXAMPLES.get(lang_code, [])]
    return "\n\n".join(lines)


THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
REASONING_MARKERS = re.compile(
    r"^\s*(here'?s a thinking process|let'?s (think|analyze)|step \d|\d+\.\s+\*\*)",
    re.IGNORECASE,
)


def clean_translation(raw_out):
    out = THINK_RE.sub("", raw_out).strip()
    return out.strip('"').strip("'").strip()


def translate_item(client, model, text, lang_code, extra_body=None, temperature=TEMPERATURE, max_retries=5):
    lang_name = LANGUAGES[lang_code]
    system = SYSTEM_PROMPT.format(lang_name=lang_name, examples_block=build_examples_block(lang_code))
    messages = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature, extra_body=extra_body or {},
            )
            out = clean_translation(resp.choices[0].message.content)
            if out and not REASONING_MARKERS.search(out):
                return out
            last_err = RuntimeError(f"looks like leaked reasoning: {out[:120]!r}")
        except Exception as e:  # noqa: BLE001
            last_err = e
        time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Translation failed for {text!r}: {last_err}")


### Checkpointed, concurrent per-item translation of a whole split

Translates both the dialog messages and the title for every row, in one pass -- each
unit is tagged `(dialog_idx, kind, turn_idx)` (`kind` is `"title"` or `"message"`) so
titles and messages share one checkpoint file per split/model.


In [7]:
def translate_mantis_split(split_name, dataset, lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    units = []
    for dialog_idx, row in enumerate(dataset):
        units.append((dialog_idx, "title", 0, row["title"]))
        for turn_idx, turn in enumerate(row["dialog"]):
            units.append((dialog_idx, "message", turn_idx, turn["message"]))

    out_path = OUT_DIR / f"{split_name}_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[(row["dialog_idx"], row["kind"], row["turn_idx"])] = row["translated"]

    todo = [u for u in units if (u[0], u[1], u[2]) not in done]
    print(f"[{split_name}/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_item, client, model, text, lang_code, extra_body): (dialog_idx, kind, turn_idx)
                for dialog_idx, kind, turn_idx, text in todo
            }
            bar = tqdm(as_completed(futures), total=len(futures), desc=f"{model_key}: {split_name}", position=position, leave=True)
            for fut in bar:
                key = futures[fut]
                translated = fut.result()
                row = {"dialog_idx": key[0], "kind": key[1], "turn_idx": key[2], "translated": translated}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                done[key] = translated

    return done


## Smoke test

Translate a handful of dialogs (title + all their messages) with both models before committing to the full run.

In [8]:
sample = raw["dev"].select(range(3))
for lang_code in LANGUAGES:
    for model_key in MODELS:
        result = translate_mantis_split("smoketest", sample, lang_code, model_key)
        for dialog_idx, row in enumerate(sample):
            print(f"[{lang_code}/{model_key}] TITLE: {row['title']!r} -> {result[(dialog_idx, 'title', 0)]!r}")
            for turn_idx, turn in enumerate(row["dialog"][:2]):
                print(f"  [{turn['role']}] {turn['message'][:100]!r} -> {result[(dialog_idx, 'message', turn_idx)][:100]!r}")
        print()


[smoketest/es/gemma] 20 cached, 0 to translate via google/gemma-4-31B-it
[es/gemma] TITLE: 'Early 2011 Macbook Pro display crash' -> 'Fallo de pantalla en MacBook Pro Early 2011'
  [user] 'In the last week I started seeing a serious issue with the display on my Macbook Pro (Early 2011, El' -> 'En la última semana empecé a notar un problema grave con la pantalla de mi Macbook Pro (Early 2011, '
  [assistant] 'This is kind of related to <a href="https://apple.stackexchange.com/questions/175465/macbook-pro-ref' -> 'Esto está relacionado con <a href="https://apple.stackexchange.com/questions/175465/macbook-pro-refu'
[es/gemma] TITLE: 'CalDAV only works for first synchronisation, and then stops updateing the calender forever' -> 'CalDAV solo funciona para la primera sincronización y luego deja de actualizar el calendario para siempre'
  [user] "From the same server I can successfully sync cardDAVs contacts, and IMAPs-mail, so I've ruled out ne" -> 'Desde el mismo servidor puedo sincronizar 

## Full run

Same gemma/qwen-in-parallel pattern as the earlier notebooks: each model works through
`SPLITS_TO_RUN` in order (dev/test first, train last) on its own thread, with its own
progress-bar row.


In [ ]:
def run_model_jobs(model_key, position):
    results = {}
    for split_name in SPLITS_TO_RUN:
        for lang_code in LANGUAGES:
            results[(split_name, lang_code, model_key)] = translate_mantis_split(
                split_name, raw[split_name], lang_code, model_key, position=position
            )
    return results


translated = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    futures = {
        ex.submit(run_model_jobs, model_key, position): model_key
        for position, model_key in enumerate(MODELS)
    }
    for fut in as_completed(futures):
        translated.update(fut.result())


[dev/es/gemma] 5584 cached, 68776 to translate via google/gemma-4-31B-it


gemma: dev:   0%|          | 0/68776 [00:00<?, ?it/s]

## Assemble translated datasets (4 columns on `dialog` turns: `role` + `message`, plus `title`, `category`, `dialog_time`)

One dataset per model, since gemma/qwen translate independently -- `category` and
`dialog_time` are copied through unchanged (not natural language).


In [ ]:
final = {}
for lang_code in LANGUAGES:
    for model_key in MODELS:
        dd = DatasetDict()
        for split_name in SPLITS_TO_RUN:
            lookup = translated[(split_name, lang_code, model_key)]
            rows = []
            for dialog_idx, row in enumerate(raw[split_name]):
                dialog = [
                    {"role": turn["role"], "message": lookup[(dialog_idx, "message", turn_idx)]}
                    for turn_idx, turn in enumerate(row["dialog"])
                ]
                rows.append({
                    "dialog": dialog,
                    "title": lookup[(dialog_idx, "title", 0)],
                    "category": row["category"],
                    "dialog_time": row["dialog_time"],
                })
            dd[split_name] = Dataset.from_list(rows)
        final[(lang_code, model_key)] = dd
        print(lang_code, model_key, dd)


## Spot-check quality

In [ ]:
lang_code = "es"
model_key = "gemma"
split_name = SPLITS_TO_RUN[0]
idxs = random.sample(range(len(final[(lang_code, model_key)][split_name])), 3)
for i in idxs:
    row = final[(lang_code, model_key)][split_name][i]
    print("TITLE:", row["title"])
    for turn in row["dialog"][:2]:
        print(f"  [{turn['role']}]", turn["message"][:200])
    print()


## Save

Saves one directory per (language, model). Pushing to the Hub is left commented out --
uncomment and set your own repo id if you want to publish.


In [ ]:
SAVE_DIR = Path("translations/mantis_final")
for (lang_code, model_key), dd in final.items():
    dd.save_to_disk(str(SAVE_DIR / f"{lang_code}-{model_key}"))

# repo_id = "<your-username>/mantis-mt"
# for (lang_code, model_key), dd in final.items():
#     dd.push_to_hub(repo_id, config_name=f"{lang_code}-{model_key}")
